# Bottom-Left法を実装する

In [1]:
from pathlib import Path 
import random
import math

class Env:

    def __init__(self, input_txt_path: Path):
        self.W, self.D, self.N, self.a = self._input(input_txt_path)

    def _input(self, txt_path):
        with open(txt_path, mode="r") as file:
            lines = file.readlines()
        W, D, N = map(int, lines[0].split())
        a = []
        for line, d in zip(lines[1:], range(D)):
            d = list(map(int, line.split()))
            a.append(d)
        return W, D, N, a

In [2]:
sample_txt_path = Path("tools/in/0001.txt")
env = Env(sample_txt_path)

In [3]:
def judge_overlap(x1, y1, x2, y2, x3, y3, x4, y4):
    return (max(x1, x3) < min(x2, x4)) and (max(y1, y3) < min(y2, y4))

class Rectangle:
    def __init__(self, target_area=None, width=None, height=None, x1=None, y1=None, x2=None, y2=None, area=None):
        self.target_area = target_area
        self.area = area
        self.width = width
        self.height = height
        self.x1 = x1
        self.y1 = y1
        self.x2 = x2
        self.y2 = y2
    
    def set_xywh(self, x1, y1, w, h):
        self.x1 = x1
        self.y1 = y1
        self.width = w
        self.height = h
        self.x2 = x1 + w
        self.y2 = y1 + h
        self.area = w * h
        self.rect_check()
    
    def x_right_plus(self, plus_w):
        self.x2 += plus_w
        self.width += plus_w
        self.area = self.width * self.height
        self.rect_check()

    def x_left_plus(self, plus_w):
        self.x1 -= plus_w
        self.width += plus_w
        self.area = self.width * self.height
        self.rect_check()
    
    def y_up_plus(self, plus_h):
        self.y1 -= plus_h
        self.height += plus_h
        self.area = self.width * self.height
        self.rect_check()
    
    def y_bottom_plus(self, plus_h):
        self.y2 += plus_h
        self.height += plus_h
        self.area = self.width * self.height
        self.rect_check()

    def rect_check(self):
        assert 0 <= self.x1 <= self.x2 <= env.W
        assert 0 <= self.y1 <= self.y2 <= env.W
        assert self.x1 + self.width == self.x2
        assert self.y1 + self.height == self.y2

    def __str__(self):
        return f"area: {self.area}, width: {self.width}, height: {self.height}, x1: {self.x1}, y1: {self.y1}, x2: {self.x2}, y2: {self.y2}"

    def judge_overlap(self, other:"Rectangle"):
        return judge_overlap(self.x1, self.y1, self.x2, self.y2, other.x1, other.y1, other.x2, other.y2)

In [4]:
import pickle

def fastcopy(obj):
    return pickle.loads(pickle.dumps(obj, -1))

In [5]:
div_n = math.ceil(env.N ** 0.5) + 1
div_coordinates = []
for d in range(div_n):
    div_coordinates.append(round(env.W / div_n * d))
div_coordinates = div_coordinates[1:]
init_coordinates = []
for x1 in div_coordinates:
    for y1 in div_coordinates:
        init_coordinates.append((x1, y1))

rectangles = []
for area, init_coordinate in zip(env.a[0], init_coordinates):
    x1, y1 = init_coordinate
    rect = Rectangle(target_area=area)
    rect.set_xywh(x1, y1, 1, 1)
    rectangles.append(rect)

In [6]:
INF = 10**9

def evaluate_cost(now_rects: list[Rectangle]):
    cost = 0
    for rect in now_rects:
        if rect.area < rect.target_area:
            cost += (rect.target_area - rect.area) * 100
    return cost

def change_action(now_rects: list[Rectangle]):
    ret_rects: list[Rectangle] = fastcopy(now_rects)
    random_ind = random.randint(0, env.N - 1)
    now_rect = ret_rects[random_ind]

    min_x_right = env.W - now_rect.x2
    min_x_left = now_rect.x1
    min_y_up = env.W - now_rect.y2
    min_y_bottom = now_rect.y1
    for i in range(env.N):
        if i == random_ind:
            continue 
        tmp_rect = ret_rects[i]
        # if now_rect.y1 <= tmp_rect.y1 and tmp_rect.y2 <= now_rect.y2:
        if now_rect.x2 <= tmp_rect.x1:
            min_x_right = min(min_x_right, tmp_rect.x1 - now_rect.x2)
        if tmp_rect.x2 <= now_rect.x1:
            min_x_left = min(min_x_left, now_rect.x1 - tmp_rect.x2)
        # if now_rect.x1 <= tmp_rect.x1 and tmp_rect.x2 <= now_rect.x2:
        if now_rect.y2 <= tmp_rect.y1:
            min_y_up = min(min_y_up, tmp_rect.y1 - now_rect.y2)
        if tmp_rect.y2 <= now_rect.y1:
            min_y_bottom = min(min_y_bottom, now_rect.y1 - tmp_rect.y2)

    select_direct = random.randint(0, 3)

    if select_direct == 0:
        # 右に伸ばす
        if min_x_right == 0:
            return ret_rects
        random_shift = random.randint(1, min(50, min_x_right))
        # change_x1 = now_rect.x2
        # change_x2 = now_rect.x2 + random_shift
        # change_y1 = now_rect.y1
        # change_y2 = now_rect.y2
        
        ret_rects[random_ind].x_right_plus(random_shift)
        # for i in range(env.N):
        #     if i == random_ind:
        #         continue 
        #     tmp_rect = ret_rects[i]
            # if judge_overlap(tmp_rect.x1, tmp_rect.y1, tmp_rect.x2, tmp_rect.y2, change_x1, change_y1, change_x2, change_y2):
            #     shift_x = tmp_rect.x1 - change_x2
            #     ret_rects[i].x_left_plus(-shift_x)
    elif select_direct == 1:
        # 左に伸ばす
        if min_x_left == 0:
            return ret_rects
        random_shift = random.randint(1, min(50, min_x_left))
        # change_x1 = now_rect.x1 - random_shift
        # change_x2 = now_rect.x1
        # change_y1 = now_rect.y1
        # change_y2 = now_rect.y2
        
        ret_rects[random_ind].x_left_plus(random_shift)
        for i in range(env.N):
            if i == random_ind:
                continue 
            tmp_rect = ret_rects[i]
            # if judge_overlap(tmp_rect.x1, tmp_rect.y1, tmp_rect.x2, tmp_rect.y2, change_x1, change_y1, change_x2, change_y2):
            #     shift_x = change_x1 - tmp_rect.x2
            #     ret_rects[i].x_right_plus(-shift_x)
    elif select_direct == 2:
        # 上に伸ばす
        if min_y_up == 0:
            return ret_rects
        random_shift = random.randint(1, min(50, min_y_up))
        # change_x1 = now_rect.x1
        # change_x2 = now_rect.x2
        # change_y1 = now_rect.y2
        # change_y2 = now_rect.y2 + random_shift
        
        ret_rects[random_ind].y_up_plus(random_shift)
        # for i in range(env.N):
        #     if i == random_ind:
        #         continue 
        #     tmp_rect = ret_rects[i]
            # if judge_overlap(tmp_rect.x1, tmp_rect.y1, tmp_rect.x2, tmp_rect.y2, change_x1, change_y1, change_x2, change_y2):
            #     shift_y = tmp_rect.y1 - change_y2
            #     ret_rects[i].y_bottom_plus(-shift_y)
    elif select_direct == 3:
        # 下に伸ばす
        if min_y_bottom == 0:
            return ret_rects
        random_shift = random.randint(1, min(50, min_y_bottom))
        # change_x1 = now_rect.x1
        # change_x2 = now_rect.x2
        # change_y1 = now_rect.y1 - random_shift
        # change_y2 = now_rect.y1
        
        ret_rects[random_ind].y_bottom_plus(random_shift)
        # for i in range(env.N):
        #     if i == random_ind:
        #         continue 
        #     tmp_rect = ret_rects[i]
            # if judge_overlap(tmp_rect.x1, tmp_rect.y1, tmp_rect.x2, tmp_rect.y2, change_x1, change_y1, change_x2, change_y2):
            #     shift_y = change_y1 - tmp_rect.y2
            #     ret_rects[i].y_up_plus(-shift_y)
    return ret_rects

In [7]:
best_x: list[Rectangle] = fastcopy(rectangles)
best_fx = evaluate_cost(best_x)

random.seed(0)
for _ in range(10000):
    change_x = change_action(best_x)
    eval_fx = evaluate_cost(change_x)
    if eval_fx < best_fx:
        best_x = change_x
        best_fx = eval_fx

AssertionError: 

In [ ]:
best_fx

98437600

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

# fig = plt.figure(figsize=(7, 7))
# ax = plt.axes()

# r = patches.Rectangle(xy=(0, 0), width=1000, height=1000, ec='#000000', fill=False)
# ax.add_patch(r)

# for x,y,w,h in zip(best_x, best_y, best_w, best_h):
#     r = patches.Rectangle(xy=(x,y), width=w, height=h, ec='#000000', fill=True)
#     ax.add_patch(r)

# plt.xlim(-5, 1500)
# plt.ylim(-5, 1500)
# plt.axis('off')
# ax.set_aspect('equal')